In [1]:
import pandas as pd

In [2]:
hotmaps = pd.read_csv('../Data/building_stock.csv', sep= "|")
hotmaps_uk = hotmaps[hotmaps['country'] == 'United Kingdom']
#hotmaps_uk = hotmaps_uk[(hotmaps_uk['topic'] == 'BUILDING') & (hotmaps_uk['type'] == 'Number of dwellings/units [Mil.]') & (hotmaps_uk['sector'] == 'Residential sector')]
hotmaps_uk = hotmaps[hotmaps['country'] == 'United Kingdom']
hotmaps_uk = hotmaps_uk[(hotmaps_uk['topic'] == 'BUILDING') 
                        & (hotmaps_uk['type'] == 'Number of dwellings/units [Mil.]') 
                        & (hotmaps_uk['subsector'] != 'Total')
                        & (hotmaps_uk['sector'] == 'Residential sector')
                        #& (hotmaps_uk['subsector'] == "Appartment blocks")
                        ]

hotmaps_uk = hotmaps_uk.dropna(subset=['bage'])
hotmaps_uk = hotmaps_uk.drop(columns=['btype', 'topic', 'feature', 'detail', 'estimated', 'source', 'unit', 'sector', 'country'])
hotmaps_uk = hotmaps_uk.reset_index(drop=True)

building_start = pd.Series([1850,1945,1970,1980,1990,2000,2010], name='building_start')
building_end = pd.Series([1944,1969,1979,1989,1999,2009,2022], name='building_end')

hotmaps_uk = pd.concat([hotmaps_uk,building_start,building_end], axis=1)

building_start = [1850,1945,1970,1980,1990,2000,2010]
building_end = [1944,1969,1979,1989,1999,2009,2022]
types = ['Single family- Terraced houses', 'Multifamily houses','Appartment blocks']
year_range = pd.Series(['1850-1944','1945-1969', '1970-1979', '1980-1989', '1990-1999', '2000-2009', '2010-2022' ], name='age_range')

new_df = pd.DataFrame()
for btype in types:
    temp_df = hotmaps_uk[hotmaps_uk['subsector'] == btype].reset_index(drop=True)

    for i in temp_df.index:

        temp_df.at[i, 'building_start'] = building_start[i]
        temp_df.at[i, 'building_end'] = building_end[i]

    temp_df['building_rate'] = temp_df['value']/(temp_df['building_end']-temp_df['building_start']+1)

    temp_df = pd.concat([temp_df, year_range], axis=1)
    new_df = pd.concat([new_df,temp_df])
        

In [7]:
new_df.to_excel('/workspaces/CUBES/exp/jack/Data/UK_Data/hotmaps_num_buildings_GB.xlsx')

In [9]:
d = {'tabula': ['1850-1918','1919-1944','1945-1964','1965-1980','1981-1990','1991-2003','2004-2009','2010-'],
        'age_range': ['0','1850-1945','1945-1969','1970-1979','1980-1989','1990-1999','2000-2010','2010-2022'],
        'Before': [0,0,0,5,0,0,0,0],
        'During': [0,26,20,10,9,9,6,13],
        'After': [69,0,0,1,1,4,0,0]}


hotmaps_tabula_age_map = pd.DataFrame(data=d)

In [10]:
hotmaps_tabula_age_map

,tabula,age_range,Before,During,After
0,1850-1918,0,0,0,69
1,1919-1944,1850-1945,0,26,0
2,1945-1964,1945-1969,0,20,0
3,1965-1980,1970-1979,5,10,1
4,1981-1990,1980-1989,0,9,1
5,1991-2003,1990-1999,0,9,4
6,2004-2009,2000-2010,0,6,0
7,2010-,2010-2022,0,13,0


In [ ]:
for index, row in hotmaps_uk.iterrows():
    country = row['country_code'].upper()
    building_type = row['btype'].upper()
    building_age = row['bage'].replace(" ","").upper
    code_end = '00'

    if "SINGLE" in building_type:
        building_type = "SFH"
    elif "MULTI" in building_type:
        building_type = "MFH"
    elif "APARTMENT" in building_type:
        building_type = "ABL"
    else:
        print("Building type not recognised.")

    if 'BEFORE' in building_age:
        building_age.replace("BEFORE", )

    building_code = country+building_type+building_age+code_end
    print(building_code)

In [ ]:
def create_ambience_building_code(hotmaps_data):
    for index, row in hotmaps_data.iterrows():
        country = row['country_code'].upper()
        building_type = row['btype']
        building_age = row['bage']
        code_end = '00'

        building_code = 

    AT-SFH-1981-1990-00
    
    country = hotmaps_data

    return hotmaps_data

In [ ]:
def add_heating_system(self):
        """Adds in thermostats for each zone and"""

        if "Boiler" in self.building_config.heating_system_type:
            heating_system_description = (
                self.building_config.heating_system_type.str.split()
            )
            energyplus_template = heating_system_description.values[0][0]
            boiler_type = heating_system_description.values[0][1]
        else:
            energyplus_template = self.building_config.heating_system_type

        for zone in self.idf.idfobjects["ZONE"]:
            stat = self.idf.newidfobject(
                "HVACTEMPLATE:THERMOSTAT",
                Name="Thermostat-" + zone.Name,
                Heating_Setpoint_Schedule_Name="Heating-Setpoint-" + zone.Name,
                Cooling_Setpoint_Schedule_Name="Cooling-Setpoint-" + zone.Name,
            )

            # if central or district heating then need radiator system in each zone
            if "Individual" not in self.building_config.heating_system_dimension:

                self.idf.newidfobject(
                    "HVACTEMPLATE:ZONE:BASEBOARDHEAT",
                    Zone_Name=zone.Name,
                    Baseboard_Heating_Type="HotWater",
                    Template_Thermostat_Name=stat.Name,
                )

        # Unsure if leaving the line below in will break the idf file if no heating
        # system is connected to the hot water loop
        self.idf.newidfobject("HVACTEMPLATE:PLANT:HOTWATERLOOP", Name="Hot Water Loop")

        if "Boiler" in energyplus_template:
            self.idf.newidfobject(
                "HVACTEMPLATE:PLANT:BOILER",
                Name="Main Boiler",
                Boiler_Type=boiler_type,
                Efficiency=self.building_config.heating_system_efficiency,
                Fuel_Type=self.building_config.heating_system_fuel,
            )

        # Unsure if need to specify plumbing for district heating
        elif "District" in energyplus_template:
            self.idf.newidfobject(energyplus_template, Name="District Heating")

        elif "Radiant" in energyplus_template:
            # EnergyPlus only can model two fuel types for radiative energy systems,
            # either electricity or natural gas for high temperature radiant systems,
            # so the fuel type is converted into natural gas if it isn't electricity

            if "Electricity" not in self.building_config.heating_system_fuel:
                self.building_config.heating_system_fuel = "NaturalGas"
            for zone in self.idf.idfobjects["ZONE"]:
                self.idf.newidfobject(
                    energyplus_template,
                    Name="Radiant Heating System",
                    Availability_schedule="Radiant-System-" + zone.Name,
                    Zone_Name=zone,
                    Fuel_Type=self.building_config.heating_system_fuel,
                    Combustion_Efficiency=(
                        self.building_config.heating_system_efficiency
                    ),
                )

        self.idf.idfobjects["SIMULATIONCONTROL"][0].Do_Zone_Sizing_Calculation = "Yes"